# 07: Case study observations (Phase 8)

**Goal:** test the guide's hypotheses for Nvidia (growth), AMD (turnaround), Intel (decline) and Wolfspeed (distress) against the computed numbers, and say plainly where the data disagrees. Every number below is computed from the project tables; the write-up is `report/drafts/case_observations.md`.

**Sources used:** Phase 3 ratios and P&L, Phase 4 Z' scores, Phase 5 walk-forward predictions, plus three items extracted here for the case companies only (capex, share buybacks, goodwill), checked against the 10-K statements. Wolfspeed's Chapter 11 dates are read from its own 10-K notes.

**Tables produced:** Table 8.1 hypothesis check, Table 8.2 warning timeline, Table 8.3 predictions against outcomes, Table 8.4 current-ratio signal, Table 8.5 growth and risk, Table 8.6 sector distress share (Tables 8.1-8.5 saved in `data/processed/`; 8.6 is `dashboard/exports/sector_by_year.csv`).

In [1]:
import os
import re
import sys
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup

sys.path.insert(0, "..")
from src import config
from src import case_studies as C
from src import verify as V

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 140)
OUT = config.PROCESSED_DIR

ratios = pd.read_csv(OUT / "ratios.csv")
zs = pd.read_csv(OUT / "zscores.csv")
mt = pd.read_csv(OUT / "modelling_table.csv")
wf = pd.read_csv(OUT / "walk_forward_predictions.csv")
num = pd.read_parquet(config.INTERIM_DIR / "num_3674.parquet")
case = C.case_rows(ratios)
case.groupby("company")["label_year"].agg(["min", "max", "count"])

,min,max,count
company,,,
AMD,2015,2025,11
Intel,2015,2025,11
Nvidia,2015,2025,11
Wolfspeed,2015,2025,11


## 1. Extra items: capex, buybacks and goodwill

Two hypotheses need cash flow items that Phase 2 did not extract: Intel's *heavy capital spending* (capital expenditure) and Nvidia's *ROE reflecting buybacks* (share repurchases). Goodwill shows the size of AMD's 2022 Xilinx acquisition. They come from the same num rows and filters as Phase 2: the filing's own fiscal year, USD, no co-registrant, no segment breakdown, full year (qtrs = 4) for cash flows and the balance sheet date (qtrs = 0) for goodwill.

**Tag check.** AMD, Intel and Wolfspeed report capex as `PaymentsToAcquirePropertyPlantAndEquipment`. Nvidia used its own custom tag `PurchasesOfPropertyAndEquipmentAndIntangibleAssets` until fiscal 2023 and the standard `PaymentsToAcquireProductiveAssets` after; both include intangible assets, so Nvidia's capex is slightly broader (flagged). A blank buyback value means the cash flow statement has no repurchase line that year; it is not filled with zero.

In [2]:
extra = C.extract_extra_items(num, ratios)
extra.to_csv(OUT / "case_extra_items.csv", index=False)
print(extra.groupby("company")[["capex_tag", "buybacks_tag", "goodwill_tag"]]
      .agg(lambda s: ", ".join(sorted(s.dropna().unique()))).to_string())
extra.pivot(index="label_year", columns="company", values=["capex", "buybacks", "goodwill"]).round(0)

                                                                                       capex_tag                        buybacks_tag goodwill_tag
company                                                                                                                                          
AMD                                                   PaymentsToAcquirePropertyPlantAndEquipment  PaymentsForRepurchaseOfCommonStock     Goodwill
Intel                                                 PaymentsToAcquirePropertyPlantAndEquipment  PaymentsForRepurchaseOfCommonStock     Goodwill
Nvidia     PaymentsToAcquireProductiveAssets, PurchasesOfPropertyAndEquipmentAndIntangibleAssets  PaymentsForRepurchaseOfCommonStock     Goodwill
Wolfspeed                                             PaymentsToAcquirePropertyPlantAndEquipment  PaymentsForRepurchaseOfCommonStock     Goodwill


capex                            buybacks                             goodwill                            
company       AMD    Intel  Nvidia Wolfspeed      AMD    Intel   Nvidia Wolfspeed      AMD    Intel   Nvidia Wolfspeed
label_year                                                                                                            
2014         95.0  10105.0   122.0     179.0      NaN  10792.0    814.0     100.0    320.0  10861.0    618.0     616.0
2015         96.0   7326.0    86.0     206.0      NaN   3001.0    587.0     550.0    278.0  11332.0    618.0     616.0
2016         77.0   9625.0   176.0     120.0      NaN   2587.0    739.0     150.0    289.0  14099.0    618.0     619.0
2017        113.0  11778.0   593.0      87.0      NaN   3615.0    909.0     104.0    289.0  24389.0    618.0     619.0
2018        163.0  15181.0   600.0     186.0      NaN  10730.0   1579.0       0.0    289.0  24513.0    618.0     620.0
2019        217.0  16213.0   489.0     142.0      NaN  13576.0      0.0       0.0    289.0  26276.0    618.0     530.0
2020        294.0  14259.0  1128.0     237.0      NaN  14229.0      0.0       NaN    289.0  26971.0   4193.0     530.0
2021        301.0  18733.0   976.0     570.0   1762.0   2415.0      NaN       NaN    289.0  26963.0   4349.0     359.0
2022        450.0  24844.0  1833.0     645.0   3702.0      0.0  10039.0       NaN  24177.0  27591.0   4372.0     359.0
2023        546.0  25750.0  1069.0     956.0    985.0      0.0   9533.0       NaN  24262.0  27591.0   4430.0     359.0
2024        636.0  23944.0  3236.0    2274.0    862.0      NaN  33706.0       NaN  24839.0  24693.0   5188.0     359.0
2025        974.0  14646.0  6042.0    1271.0   1316.0      NaN  40086.0       NaN  25126.0  23912.0  20832.0       0.0

### Check against the 10-K statements

For every case company in 2017, 2022 and 2024, each value is looked up in the 10-K's rendered cash flow statement or balance sheet (EDGAR "R" pages, cached in `data/raw/edgar_statements/`). Payments are shown in brackets in the cash flow statement, so sizes are compared. Blank items are skipped.

In [3]:
ua = os.environ.get(config.SEC_USER_AGENT_ENV, "")  # only needed if a page is not cached yet
checks = []
for _, row in extra[extra["label_year"].isin([2017, 2022, 2024])].iterrows():
    paths = V.download_statements(row["cik"], row["adsh"], config.RAW_DIR / "edgar_statements", ua)
    out = V.check_filing(row, paths, C.EXTRA_CHECK_ITEMS, absolute=True)
    out.insert(0, "label_year", row["label_year"])
    out.insert(0, "company", row["company"])
    out["filing"] = V.filing_url(row["cik"], row["adsh"], "")
    checks.append(out)
checks = pd.concat(checks, ignore_index=True)
checks.to_csv(OUT / "case_extra_verification.csv", index=False)
print(checks["status"].value_counts().to_string())
checks[["company", "label_year", "item", "computed_musd", "status", "tenk_line", "tenk_value_musd"]]

status
match    32


,company,label_year,item,computed_musd,status,tenk_line,tenk_value_musd
0,AMD,2017,capex,113.000,match,"Purchases of property, plant and equipment",113.000
1,AMD,2017,goodwill,289.000,match,Goodwill,289.000
2,AMD,2022,capex,450.000,match,Purchases of property and equipment,450.000
3,AMD,2022,buybacks,3702.000,match,Repurchases of common stock,3702.000
4,AMD,2022,goodwill,24177.000,match,Goodwill,24177.000
5,AMD,2024,capex,636.000,match,Purchases of property and equipment,636.000
6,AMD,2024,buybacks,862.000,match,Repurchases of common stock,862.000
7,AMD,2024,goodwill,24839.000,match,Goodwill,24839.000
8,Intel,2017,capex,11778.000,match,"Additions to property, plant and equipment",11778.000
9,Intel,2017,buybacks,3615.000,match,Repurchase of common stock,3615.000


## 2. Wolfspeed's Chapter 11: dates from its own filings

The guide asks to verify the restructuring dates and end Wolfspeed's series at the last reliable year. The sentences below are read from Note 2 and the subsequent-events note of the fiscal 2025 10-K, and from the basis-of-presentation note of the fiscal 2026 10-K (which is outside the analysis years).

In [4]:
def sentences(adsh, page, pattern):
    raw = (config.RAW_DIR / "edgar_statements" / adsh / page).read_bytes()
    try:
        html = raw.decode("utf-8")
    except UnicodeDecodeError:
        html = raw.decode("cp1252")  # same fallback as src/verify.parse_statement
    text = BeautifulSoup(html, "html.parser").get_text(" ", strip=True)
    # Curly quotes and dashes as plain characters (the Windows notebook output cannot show them)
    text = text.translate(str.maketrans({"\u201c": '"', "\u201d": '"', "\u2019": "'", "\u2014": "-", "\u2013": "-", "\xa0": " "}))
    return [s.strip() for s in re.findall(r"[^.]*(?:" + pattern + r")[^.]*\.", text)]

fy2025 = sentences("0000895419-25-000110", "R10.htm", r"filed voluntary petitions|presented as")
fy2026 = sentences("0000895419-26-000054", "R11.htm", r"emerged from Chapter 11|resulting in a new reporting entity|13-week fiscal period|39-week fiscal period")
for s in fy2025 + fy2026:
    print("-", s[:330], "\n")

- On June 30, 2025 (the "Petition Date"), the Debtors filed voluntary petitions (the "Chapter 11 Cases") under Chapter 11 of Title 11 of the United States Bankruptcy Code (the "Bankruptcy Code") in the United States Bankruptcy Court for the Southern District of Texas, Houston Division (the "Bankruptcy Court") to implement a Chapte 

- Based on the facts and circumstances described above, including the defaults related to the missed interest payment on the 2029 Notes that were in grace periods as of June 29, 2025, the signing of the Restructuring Support Agreement on June 23, 2025 and subsequent events of default upon filing of the Chapter 11 Cases per the ter 

- The Company's Predecessor period from June 30, 2025 to September 29, 2025 was a 13-week fiscal period. 

- The Company's Successor period from September 30, 2025 to June 28, 2026 was a 39-week fiscal period. 

- Recognition of Intangible Assets As further described below, on the Effective Date, the Company adopted fresh start 

**Verified:** the fiscal year ended 29 June 2025; Chapter 11 petitions were filed on 30 June 2025 (one day later). Because of the missed interest payment on the 2029 notes, the restructuring support agreement (23 June) and the defaults the filing triggered, the 29 June 2025 balance sheet presents the long-term borrowings as current, which is why current liabilities jump in 2025. The company emerged on 29 September 2025 and adopted fresh-start accounting, becoming a new reporting entity; its fiscal 2026 report is split into a 13-week Predecessor and a 39-week Successor period. So **label year 2025 is Wolfspeed's last comparable year**, as already used since Phase 3.

## 3. Evidence by company

The ratio table for each case company, with capex intensity and buybacks added. Money in USD millions; ratios as decimals.

In [5]:
d = (case.merge(extra[["cik", "label_year", "capex", "buybacks", "goodwill"]], on=["cik", "label_year"])
         .merge(zs[["cik", "label_year", "z_prime", "zone"]], on=["cik", "label_year"]))
d["capex_to_revenue"] = d["capex"] / d["revenue"]
d["buybacks_to_net_income"] = np.where(d["net_income"] > 0, d["buybacks"] / d["net_income"], np.nan)
COLS = ["revenue", "net_income", "gross_margin", "operating_margin", "roa", "roe", "current_ratio", "debt_ratio",
        "debt_to_equity", "interest_coverage", "total_assets", "total_liabilities", "equity", "asset_turnover",
        "capex_to_revenue", "buybacks", "goodwill", "z_prime"]
def show(company):
    return d[d["company"] == company].set_index("label_year")[COLS].round(3).T
show("Nvidia")

label_year,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
revenue,5010.000,6910.000,9714.000,11716.000,10918.000,16675.000,26914.000,26974.000,60922.000,130497.000,215938.000
net_income,614.000,1666.000,3047.000,4141.000,2796.000,4332.000,9752.000,4368.000,29760.000,72880.000,120067.000
gross_margin,0.561,0.588,0.599,0.612,0.620,0.623,0.649,0.569,0.727,0.750,0.711
operating_margin,0.149,0.280,0.330,0.325,0.261,0.272,0.373,0.157,0.541,0.624,0.604
roa,0.083,0.169,0.271,0.312,0.161,0.150,0.221,0.106,0.453,0.653,0.581
roe,0.137,0.289,0.408,0.443,0.229,0.256,0.366,0.198,0.692,0.919,0.763
current_ratio,2.575,4.774,8.027,7.944,7.674,4.090,6.650,3.516,4.171,4.440,3.905
debt_ratio,0.382,0.411,0.335,0.297,0.295,0.413,0.398,0.463,0.346,0.289,0.239
debt_to_equity,0.630,0.703,0.505,0.423,0.419,0.704,0.660,0.863,0.529,0.407,0.315
interest_coverage,15.894,33.345,52.623,65.586,54.731,24.630,42.547,16.122,128.296,329.769,503.425


In [6]:
# Why Nvidia was in the grey zone: each component's weighted contribution (they add up to Z').
zb = C.z_breakdown(zs)
nv = zb[(zb["company"] == "Nvidia") & (zb["label_year"] >= config.LABEL_YEAR_START)].set_index("label_year")
nv_z = nv[[f"X{i}_contrib" for i in range(1, 6)] + ["z_prime"]].round(2)
nv_z.columns = [C.Z_PARTS.get(c[:2], c) if c != "z_prime" else "Z'" for c in nv_z.columns]
nv_z

,Working capital / assets,Retained earnings / assets,Operating income / assets,Book equity / liabilities,Revenue / assets,Z'
label_year,,,,,,
2015,0.36,0.50,0.31,0.67,0.68,2.52
2016,0.49,0.53,0.61,0.60,0.70,2.93
2017,0.52,0.66,0.89,0.83,0.86,3.76
2018,0.50,0.80,0.89,0.99,0.88,4.06
2019,0.49,0.73,0.51,1.00,0.63,3.37
2020,0.30,0.56,0.49,0.60,0.58,2.52
2021,0.40,0.31,0.71,0.64,0.61,2.66
2022,0.29,0.21,0.32,0.49,0.65,1.96
2023,0.37,0.38,1.56,0.79,0.93,4.03


In [7]:
# Nvidia's retained earnings fell while it was profitable. The equity statement explains the two drops:
# 2021: treasury stock (earlier buybacks) retired against retained earnings; 2022: buybacks retired directly.
nv_adsh = ratios.loc[(ratios["cik"] == 1045810) & (ratios["label_year"] == 2021), "adsh"].iloc[0]
retired = num[(num["adsh"] == nv_adsh) & (num["tag"] == "TreasuryStockRetiredCostMethodAmount") & (num["qtrs"] == 4)]
print(retired[["tag", "segments", "value"]].assign(value=lambda x: x["value"] / 1e6).to_string(index=False))
re_ = ratios[ratios["cik"] == 1045810].set_index("label_year")[["retained_earnings", "net_income"]]
re_["buybacks"] = extra[extra["company"] == "Nvidia"].set_index("label_year")["buybacks"]
re_.loc[2019:2023]

                                 tag                                  segments    value
TreasuryStockRetiredCostMethodAmount        EquityComponents=RetainedEarnings;  12026.0
TreasuryStockRetiredCostMethodAmount EquityComponents=AdditionalPaidInCapital;     20.0
TreasuryStockRetiredCostMethodAmount           EquityComponents=TreasuryStock; -12046.0
TreasuryStockRetiredCostMethodAmount                                       NaN      0.0


,retained_earnings,net_income,buybacks
label_year,,,
2019,14971.0,2796.0,0.0
2020,18908.0,4332.0,0.0
2021,16235.0,9752.0,NaN
2022,10171.0,4368.0,10039.0
2023,29817.0,29760.0,9533.0


In [8]:
# DuPont: ROE = net margin x asset turnover x equity multiplier. A low multiplier means ROE is not
# inflated by a thin equity base.
dp = C.dupont(case)
print("max |ROE - product| =", float((dp["roe"] - dp["roe_check"]).abs().max()))
dp[dp["company"] == "Nvidia"].set_index("label_year")[["net_margin", "asset_turnover", "equity_multiplier", "roe"]].round(3).T

max |ROE - product| = 2.220446049250313e-16


label_year,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
net_margin,0.123,0.241,0.314,0.353,0.256,0.260,0.362,0.162,0.488,0.558,0.556
asset_turnover,0.680,0.702,0.864,0.881,0.631,0.579,0.609,0.655,0.927,1.169,1.044
equity_multiplier,1.649,1.708,1.505,1.423,1.419,1.704,1.660,1.863,1.529,1.407,1.315
roe,0.137,0.289,0.408,0.443,0.229,0.256,0.366,0.198,0.692,0.919,0.763


In [9]:
show("AMD")

label_year,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
revenue,3991.000,4272.000,5329.000,6475.000,6731.000,9763.000,16434.000,23601.000,22680.000,25785.000,34639.000
net_income,-660.000,-497.000,43.000,337.000,341.000,2490.000,3162.000,1320.000,854.000,1641.000,4335.000
gross_margin,0.271,0.234,0.342,0.378,0.426,0.445,0.482,0.449,0.461,0.494,0.495
operating_margin,-0.121,-0.087,0.038,0.070,0.094,0.140,0.222,0.054,0.018,0.074,0.107
roa,-0.212,-0.150,0.012,0.074,0.057,0.278,0.255,0.020,0.013,0.024,0.056
roe,NaN,-1.195,0.070,0.266,0.121,0.427,0.422,0.024,0.015,0.029,0.069
current_ratio,1.654,1.880,1.764,1.784,1.949,2.542,2.024,2.358,2.507,2.616,2.850
debt_ratio,1.133,0.875,0.827,0.722,0.531,0.349,0.396,0.190,0.177,0.168,0.181
debt_to_equity,NaN,6.983,4.794,2.599,1.132,0.535,0.657,0.234,0.215,0.203,0.221
interest_coverage,-3.006,-2.385,1.619,3.727,6.713,29.128,107.294,14.364,3.783,20.652,28.198


In [10]:
show("Intel")

label_year,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
revenue,55355.000,59387.000,62761.000,70848.000,71965.000,77867.000,79024.000,63054.000,54228.000,53101.000,52853.000
net_income,11420.000,10316.000,9601.000,21053.000,21048.000,20899.000,19868.000,8014.000,1689.000,-18756.000,-267.000
gross_margin,0.626,0.609,0.623,0.617,0.586,0.560,0.554,0.426,0.400,0.327,0.348
operating_margin,0.253,0.217,0.286,0.329,0.306,0.304,0.246,0.037,0.002,-0.220,-0.042
roa,0.111,0.091,0.078,0.165,0.154,0.137,0.118,0.044,0.009,-0.095,-0.001
roe,0.187,0.156,0.139,0.282,0.272,0.258,0.208,0.079,0.016,-0.189,-0.002
current_ratio,2.576,1.749,1.693,1.731,1.400,1.909,2.102,1.568,1.542,1.327,2.017
debt_ratio,0.399,0.408,0.433,0.414,0.431,0.471,0.434,0.433,0.426,0.465,0.402
debt_to_equity,0.673,0.698,0.773,0.711,0.760,0.889,0.765,0.777,0.773,0.921,0.744
interest_coverage,41.549,17.563,27.765,49.821,45.061,37.644,32.590,4.706,0.106,-11.294,-2.029


In [11]:
show("Wolfspeed")

label_year,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
revenue,1632.505,1616.627,1473.000,1493.680,1080.000,903.900,525.600,746.200,921.900,807.200,757.600
net_income,-64.051,-21.536,-98.118,-279.968,-375.100,-191.700,-523.900,-200.900,-329.900,-864.200,-1609.200
gross_margin,0.291,0.301,0.295,0.273,0.362,0.275,0.313,0.334,0.303,0.096,-0.161
operating_margin,-0.044,-0.006,-0.013,-0.220,-0.015,-0.232,-0.597,-0.332,-0.413,-0.552,-1.754
roa,-0.022,-0.008,-0.037,-0.106,-0.133,-0.059,-0.152,-0.051,-0.050,-0.108,-0.235
roe,-0.026,-0.009,-0.044,-0.135,-0.184,-0.092,-0.248,-0.082,-0.203,-0.980,NaN
current_ratio,5.135,5.187,5.191,3.585,5.268,5.470,3.295,4.536,5.744,4.509,0.359
debt_ratio,0.165,0.144,0.161,0.214,0.275,0.353,0.386,0.377,0.754,0.890,1.065
debt_to_equity,0.198,0.168,0.192,0.274,0.381,0.548,0.629,0.606,3.061,8.052,NaN
interest_coverage,-72.368,-3.367,-5.204,-54.011,-3.975,-6.000,-6.914,-9.873,-8.934,-1.808,-4.217


## 4. Table 8.1: hypothesis check

Each hypothesis from the guide's Phase 8 table, the numbers that test it, and a verdict. The verdict rule is written next to each row so it can be challenged: *Supported* when the numbers show the pattern, *Partly* when only some of it holds, *Contradicted* when the numbers show the opposite.

In [12]:
g = {c: d[d["company"] == c].set_index("label_year") for c in config.CASE_COMPANIES.values()}
nvd, amd, itl, wsp = g["Nvidia"], g["AMD"], g["Intel"], g["Wolfspeed"]
pct = lambda v: f"{v * 100:.0f}%"
nv_em = dp[dp["company"] == "Nvidia"]["equity_multiplier"]
nv_retired = retired.loc[retired["segments"].fillna("").str.contains("RetainedEarnings"), "value"].iloc[0] / 1e6
H = []
def add(company, hypothesis, evidence, verdict, source):
    H.append({"company": company, "hypothesis": hypothesis, "evidence": evidence, "verdict": verdict, "source": source})

# Nvidia
add("Nvidia", "Revenue, margins, ROA and ROE rise sharply",
    f"Revenue {nvd.revenue[2015]:,.0f} -> {nvd.revenue[2025]:,.0f}; operating margin {pct(nvd.operating_margin[2015])} -> {pct(nvd.operating_margin[2024])} (2024); "
    f"ROE {pct(nvd.roe[2015])} -> {pct(nvd.roe[2024])} (2024). Not a smooth rise: dips in 2019 (operating margin {pct(nvd.operating_margin[2019])}) and 2022 ({pct(nvd.operating_margin[2022])}).",
    "Supported", "Figures 2, 6, 7")
add("Nvidia", "Liquidity stays strong",
    f"Current ratio between {nvd.current_ratio.min():.2f} and {nvd.current_ratio.max():.2f} in every year.",
    "Supported" if nvd.current_ratio.min() > 1.5 else "Partly", "Figure 4")
grey = [int(y) for y in nvd.index[nvd.z_prime < 2.90]]
add("Nvidia", "Z' stays far above 2.90",
    f"Z' below 2.90 (grey zone) in {', '.join(map(str, grey))}; lowest {nvd.z_prime.min():.2f} in {int(nvd.z_prime.idxmin())}. Causes: revenue/assets (X5) below 1 until 2024, and retained earnings cut by retiring bought-back shares "
    f"({nv_retired:,.0f} in 2021, buybacks {nvd.buybacks[2022]:,.0f} in 2022), so X2 fell from {zb.set_index(['company','label_year']).loc[('Nvidia',2018),'X2_contrib']:.2f} (2018) to {zb.set_index(['company','label_year']).loc[('Nvidia',2022),'X2_contrib']:.2f} (2022).",
    "Contradicted" if grey else "Supported", "Figure 8; Z' breakdown (section 3)")
add("Nvidia", "Very high ROE can reflect buybacks as well as profit",
    f"DuPont 2024: net margin {nvd.net_margin[2024]:.2f} x asset turnover {nvd.asset_turnover[2024]:.2f} x equity multiplier {dp.set_index(['company','label_year']).loc[('Nvidia',2024),'equity_multiplier']:.2f} = ROE {nvd.roe[2024]:.2f}. "
    f"Multiplier only {nv_em.min():.2f}-{nv_em.max():.2f}, so ROE is driven by profit; buybacks were large ({nvd.buybacks[2024]:,.0f} in 2024, {pct(nvd.buybacks_to_net_income[2024])} of net income) but equity still grew {nvd.equity[2023]:,.0f} -> {nvd.equity[2024]:,.0f}.",
    "Partly", "DuPont table (section 3)")
# AMD
add("AMD", "Weak profitability and thin equity in the early years",
    f"Losses in 2015-2016 (net income {amd.net_income[2015]:,.0f}, {amd.net_income[2016]:,.0f}); equity {amd.equity[2015]:,.0f} (negative) in 2015, debt-to-equity {amd.debt_to_equity[2016]:.2f} in 2016; Z' {amd.z_prime[2015]:.2f}.",
    "Supported", "Figures 3, 5, 8")
add("AMD", "Then recovery and growth",
    f"Revenue {amd.revenue[2016]:,.0f} (2016) -> {amd.revenue[2025]:,.0f} (2025); debt-to-equity fell to {amd.debt_to_equity[2025]:.2f}; Z' rose above 1.23 in 2020 ({amd.z_prime[2020]:.2f}) and stayed in the grey zone or above.",
    "Supported", "Figures 2, 5, 8")
add("AMD", "Xilinx (2022) raises assets and goodwill, lowering ROA and asset turnover",
    f"Total assets {amd.total_assets[2021]:,.0f} -> {amd.total_assets[2022]:,.0f}; goodwill {amd.goodwill[2021]:,.0f} -> {amd.goodwill[2022]:,.0f}; asset turnover {amd.asset_turnover[2021]:.2f} -> {amd.asset_turnover[2022]:.2f}; ROA {pct(amd.roa[2021])} -> {pct(amd.roa[2022])}. "
    f"Z' fell {amd.z_prime[2021]:.2f} -> {amd.z_prime[2022]:.2f} through X5, without any distress.",
    "Supported", "Figure 7; AMD table (section 3)")
# Intel
add("Intel", "Falling margins",
    f"Gross margin {pct(itl.gross_margin[2015])} (2015) -> {pct(itl.gross_margin[2024])} (2024); operating margin {pct(itl.operating_margin[2021])} (2021) -> {pct(itl.operating_margin[2022])} (2022) -> {pct(itl.operating_margin[2024])} (2024).",
    "Supported", "Figure 6")
add("Intel", "Heavy capital spending",
    f"Capex {itl.capex[2015]:,.0f} ({pct(itl.capex_to_revenue[2015])} of revenue, 2015) -> {itl.capex[2023]:,.0f} ({pct(itl.capex_to_revenue[2023])}, 2023) while revenue fell from {itl.revenue[2021]:,.0f} (2021) to {itl.revenue[2023]:,.0f}; buybacks stopped (0 in 2022-2023, no line after).",
    "Supported", "Table 8.1 (capex, verified against 10-K)")
add("Intel", "Rising liabilities",
    f"Total liabilities {itl.total_liabilities[2015]:,.0f} -> {itl.total_liabilities[2024]:,.0f} (2024), but debt ratio stayed {itl.debt_ratio.min():.2f}-{itl.debt_ratio.max():.2f} because equity also grew; debt-to-equity peaked at {itl.debt_to_equity.max():.2f} (2024).",
    "Partly", "Figure 5")
add("Intel", "Net loss around 2024",
    f"Net income {itl.net_income[2023]:,.0f} (2023) -> {itl.net_income[2024]:,.0f} (2024) -> {itl.net_income[2025]:,.0f} (2025).",
    "Supported", "Figure 3")
add("Intel", "Leverage and coverage worsen before profit collapses",
    f"Coverage fell {itl.interest_coverage[2021]:.1f} (2021) -> {itl.interest_coverage[2022]:.1f} (2022) -> {itl.interest_coverage[2023]:.2f} (2023), one to two years before the 2024 loss; Z' < 1.23 from 2023. "
    f"The fall came from operating income ({itl.operating_income[2021]:,.0f} -> {itl.operating_income[2022]:,.0f} -> {itl.operating_income[2023]:,.0f}), not interest ({itl.interest_expense.loc[2021:2023].min():,.0f}-{itl.interest_expense.loc[2021:2023].max():,.0f}). Leverage gave no warning (debt ratio never above 0.6).",
    "Partly", "Figure 5; Table 8.2")
# Wolfspeed
add("Wolfspeed", "Persistent losses",
    f"Net loss in all 11 years (2015 {wsp.net_income[2015]:,.0f} to 2025 {wsp.net_income[2025]:,.0f}); operating margin negative every year, {pct(wsp.operating_margin[2025])} in 2025.",
    "Supported" if (wsp.net_income < 0).all() else "Partly", "Figures 3, 6")
add("Wolfspeed", "High debt",
    f"Debt ratio {wsp.debt_ratio[2015]:.2f} (2015) -> {wsp.debt_ratio[2023]:.2f} (2023) -> {wsp.debt_ratio[2025]:.2f} (2025); debt-to-equity {wsp.debt_to_equity[2024]:.2f} (2024); equity negative in 2025. Low until 2022.",
    "Supported (from 2023)", "Figure 5")
add("Wolfspeed", "Weak interest coverage",
    f"Coverage negative in every year (operating losses), e.g. {wsp.interest_coverage[2024]:.2f} in 2024; 2015-2019 use cash interest paid as a proxy.",
    "Supported", "Figure 5")
add("Wolfspeed", "Weak liquidity",
    f"Current ratio {wsp.current_ratio.drop(2025).min():.2f}-{wsp.current_ratio.drop(2025).max():.2f} in 2015-2024 (still {wsp.current_ratio[2024]:.2f} in 2024), then {wsp.current_ratio[2025]:.2f} in 2025, when the 10-K presented its long-term borrowings as current "
    "because of the missed interest payment, the restructuring agreement and the defaults triggered by the Chapter 11 filing.",
    "Contradicted (until 2025)", "Figure 4")
add("Wolfspeed", "Then restructuring",
    "Chapter 11 petition 30 June 2025 (fiscal year end 29 June 2025); emerged 29 September 2025 with fresh-start accounting. Series ends at 2025.",
    "Supported (verified in 10-K notes)", "Section 2")
hyp = pd.DataFrame(H)
hyp.to_csv(OUT / "case_hypotheses.csv", index=False)
print(hyp["verdict"].value_counts().to_string())
hyp

verdict
Supported                             10
Partly                                 3
Contradicted                           1
Supported (from 2023)                  1
Contradicted (until 2025)              1
Supported (verified in 10-K notes)     1


,company,hypothesis,evidence,verdict,source
0,Nvidia,"Revenue, margins, ROA and ROE rise sharply","Revenue 5,010 -> 215,938; operating margin 15% -> 62% (2024); ROE 14% -> 92% (2024). Not a smooth rise: dips in 2019 (operating margin 2...",Supported,"Figures 2, 6, 7"
1,Nvidia,Liquidity stays strong,Current ratio between 2.57 and 8.03 in every year.,Supported,Figure 4
2,Nvidia,Z' stays far above 2.90,"Z' below 2.90 (grey zone) in 2015, 2020, 2021, 2022; lowest 1.96 in 2022. Causes: revenue/assets (X5) below 1 until 2024, and retained e...",Contradicted,Figure 8; Z' breakdown (section 3)
3,Nvidia,Very high ROE can reflect buybacks as well as profit,"DuPont 2024: net margin 0.56 x asset turnover 1.17 x equity multiplier 1.41 = ROE 0.92. Multiplier only 1.31-1.86, so ROE is driven by p...",Partly,DuPont table (section 3)
4,AMD,Weak profitability and thin equity in the early years,"Losses in 2015-2016 (net income -660, -497); equity -412 (negative) in 2015, debt-to-equity 6.98 in 2016; Z' -1.03.",Supported,"Figures 3, 5, 8"
5,AMD,Then recovery and growth,"Revenue 4,272 (2016) -> 34,639 (2025); debt-to-equity fell to 0.22; Z' rose above 1.23 in 2020 (2.21) and stayed in the grey zone or above.",Supported,"Figures 2, 5, 8"
6,AMD,"Xilinx (2022) raises assets and goodwill, lowering ROA and asset turnover","Total assets 12,419 -> 67,580; goodwill 289 -> 24,177; asset turnover 1.32 -> 0.35; ROA 25% -> 2%. Z' fell 3.02 -> 2.29 through X5, with...",Supported,Figure 7; AMD table (section 3)
7,Intel,Falling margins,Gross margin 63% (2015) -> 33% (2024); operating margin 25% (2021) -> 4% (2022) -> -22% (2024).,Supported,Figure 6
8,Intel,Heavy capital spending,"Capex 7,326 (13% of revenue, 2015) -> 25,750 (47%, 2023) while revenue fell from 79,024 (2021) to 54,228; buybacks stopped (0 in 2022-20...",Supported,"Table 8.1 (capex, verified against 10-K)"
9,Intel,Rising liabilities,"Total liabilities 41,083 -> 91,453 (2024), but debt ratio stayed 0.40-0.47 because equity also grew; debt-to-equity peaked at 0.92 (2024).",Partly,Figure 5


## 5. Table 8.2: which ratio moved first?

The first analysis year each warning sign appeared, next to the first year Z' fell below 1.23. Warning levels are fixed rule-of-thumb levels chosen before looking at the results (see `src/case_studies.py`): current ratio < 1, debt ratio > 0.6, interest coverage < 1.5, operating margin < 0, net loss. A year of 2015 means the sign was already present in the first analysis year, so no lead time can be measured.

In [13]:
tl = C.warning_timeline(case, zs)
tl.to_csv(OUT / "case_warning_timeline.csv", index=False)
tl.set_index("company").astype("Int64")

,Liquidity: current ratio < 1,Leverage: debt ratio > 0.6,Coverage: interest coverage < 1.5,Margin: operating margin < 0,Profit: net loss,First Z' < 1.23
company,,,,,,
AMD,<NA>,2015,2015,2015,2015,2015
Intel,<NA>,<NA>,2023,2024,2024,2023
Nvidia,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
Wolfspeed,2025,2023,2015,2015,2015,2020


## 6. Table 8.3: what the model predicted

Walk-forward random forest (trained only on earlier years): the year shown is the feature year t, the outcome is the Z' label in t+1. Threshold 0.494.

In [14]:
po = C.prediction_outcomes(wf)
po.to_csv(OUT / "case_prediction_outcomes.csv", index=False)
print(po.pivot(index="label_year", columns="company", values="prob_distressed_t1").round(2)[list(config.CASE_COMPANIES.values())].to_string())
po.pivot(index="label_year", columns="company", values="outcome")[list(config.CASE_COMPANIES.values())]

company     Nvidia   AMD  Intel  Wolfspeed
label_year                                
2018          0.15  0.35   0.14       0.78
2019          0.10  0.32   0.17       0.76
2020          0.07  0.11   0.18       0.73
2021          0.12  0.07   0.13       0.94
2022          0.17  0.11   0.28       0.83
2023          0.11  0.14   0.67       0.88
2024          0.09  0.10   0.85       0.84
2025          0.09  0.06   0.73       0.89


company,Nvidia,AMD,Intel,Wolfspeed
label_year,,,,
2018,correct (healthy),missed,correct (healthy),false alarm
2019,correct (healthy),correct (healthy),correct (healthy),correct warning
2020,correct (healthy),correct (healthy),correct (healthy),correct warning
2021,correct (healthy),correct (healthy),correct (healthy),correct warning
2022,correct (healthy),correct (healthy),missed,correct warning
2023,correct (healthy),correct (healthy),correct warning,correct warning
2024,correct (healthy),correct (healthy),correct warning,correct warning
2025,forecast (t+1 not yet known),forecast (t+1 not yet known),forecast (t+1 not yet known),forecast (t+1 not yet known)


## 7. Cross-company and sector checks

**Table 8.4: is a current ratio below 1 a false alarm?** The guide expects single-ratio rules to give false alarms (a low current ratio that does not lead to distress). One-year-ahead labelled rows, 2015-2024 feature years.

In [15]:
crs = C.current_ratio_signal(mt)
crs.to_csv(OUT / "current_ratio_signal.csv", index=False)
crs

,scope,company_years,current_ratio_below_1,of_which_distressed_next_year,false_alarms,distressed_next_year,distressed_with_current_ratio_at_least_1
0,Operating companies,670,15,13,2,249,236
1,All companies,748,39,37,2,318,281


**Table 8.5: does high growth go with low risk?** Operating companies split into four equal groups by revenue growth in year t; the share distressed in year t+1.

In [16]:
gr = C.growth_vs_risk(mt)
gr.to_csv(OUT / "growth_vs_risk.csv", index=False)
gr.round(3)

,growth_quartile,company_years,growth_min,growth_max,median_net_margin,share_distressed_next_year
0,Q1 (lowest),158,-0.697,-0.065,-0.067,0.468
1,Q2,158,-0.065,0.061,0.055,0.316
2,Q3,157,0.062,0.235,0.083,0.318
3,Q4 (highest),158,0.237,2.062,0.073,0.342


**Table 8.6: sector distress share by year.** Share of company-years with Z' < 1.23 (same values as the dashboard reference table).

In [17]:
# Sector: share of companies with Z' < 1.23 by year (same values as the dashboard reference table).
sector = pd.read_csv(config.DASHBOARD_EXPORTS_DIR / "sector_by_year.csv")
sector.pivot(index="label_year", columns="scope", values="pct_distressed").mul(100).round(1)

scope,All companies,Operating companies (revenue >= USD 10m)
label_year,,
2015,30.6,28.2
2016,49.4,46.6
2017,45.7,39.7
2018,40.2,34.2
2019,46.9,41.4
2020,39.7,35.3
2021,44.7,37.0
2022,37.8,30.6
2023,40.0,33.3
